# Speech tutorial: QC, transcripts, and linguistic markers

Rudzicz Lab — Centre for Analytics (CfA)

This notebook walks through the **voice** processing used alongside the actigraphy relapse model on CBN-WELL.

It runs on synthetic SRI-shaped QC rows and invented transcripts in `test_data/`. There is **no audio** and no Whisper call.

**Goal:** show the same analysis path as the original work — task QC, then transcripts, then exploratory linguistic markers — with **negation ratio** as the standout speech signal.

## 1. Speech data and tasks

The protocol recordings that matter for this analysis are:

- **Sustained vowel** — /a/ for at least 5 seconds (voice quality)
- **Free speech** — describe physical condition, mental condition, or a happy event (at least 30 seconds)

Read speech and automatic speech (counting / alphabet) were also collected. They are included in QC below so the gates stay complete; they are not the linguistic-marker stream.

QC uses SRI columns: `SADSPEECHEXISTS`, `SIGNALSNR`, `PERCENTCLIPPED`, `DURATIONSECONDS`, `MULTISPEAKER`, `SIGNALRMS`.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "test_data").exists():
    ROOT = Path("Rudzicz-Lab")
DATA = ROOT / "test_data"

qc = pd.read_csv(DATA / "sample_qc.csv")
transcripts = pd.read_csv(DATA / "sample_transcripts.csv")
qc

## 2. Quality control

SAD looks for *speech*, not a held vowel, so it is **not** a vowel criterion. A vowel file with `SADSPEECHEXISTS == 0` can still be valid if RMS, SNR, clipping, and duration pass.

| Task | SAD | Duration | SNR | Clipping | Extra |
|------|-----|----------|-----|----------|--------|
| Sustained vowel | **ignored** | ≥ 5 s | ≥ 10 dB | ≤ 5% | RMS ≥ 500 |
| Free speech | required | ≥ 30 s | ≥ 10 dB | ≤ 10% | single speaker |
| Read / counting | required | ≥ 20 s (and ≤ 120 s) | ≥ 10 dB | ≤ 10% | single speaker |

Acoustic features (including F0) were extracted only on files that pass these gates. The cells below use synthetic QC rows.

In [5]:
def vowel_ok(row):
    # SADSPEECHEXISTS is intentionally unused for sustained vowel.
    return (
        row["SIGNALRMS"] >= 500
        and row["SIGNALSNR"] >= 10
        and row["PERCENTCLIPPED"] <= 5
        and row["DURATIONSECONDS"] >= 5
    )


def speech_task_ok(row, min_dur, max_dur=None):
    ok = (
        row["SADSPEECHEXISTS"] == 1
        and row["SIGNALSNR"] >= 10
        and row["PERCENTCLIPPED"] <= 10
        and row["MULTISPEAKER"] == 0
        and row["DURATIONSECONDS"] >= min_dur
    )
    if max_dur is not None:
        ok = ok and row["DURATIONSECONDS"] <= max_dur
    return bool(ok)


def recording_ok(row):
    task = row["task"]
    if task == "SustainedVowel":
        return vowel_ok(row)
    if task in {"ReadSpeech", "AutomaticSpeech"}:
        return speech_task_ok(row, min_dur=20, max_dur=120)
    if task in {"PhysicalCondition", "MentalCondition", "HappyEvent"}:
        return speech_task_ok(row, min_dur=30)
    return False


qc = qc.copy()
qc["valid"] = qc.apply(recording_ok, axis=1)
qc["sad_only"] = qc["SADSPEECHEXISTS"] == 1

# SAD is not a vowel criterion: SYN_001 vowel has SAD=0 and should still be valid.
# SYN_002 vowel fails on RMS, not on SAD.
print(qc[["FILE", "task", "SADSPEECHEXISTS", "SIGNALRMS", "SIGNALSNR", "sad_only", "valid"]].to_string(index=False))
print()
print("valid by task:")
print(qc.groupby("task")["valid"].agg(["sum", "count"]))

                                 FILE              task  SADSPEECHEXISTS  SIGNALRMS  SIGNALSNR  sad_only  valid
       20180501_SYN001_ReadSpeech.wav        ReadSpeech                1       1400         18      True   True
  20180501_SYN001_AutomaticSpeech.wav   AutomaticSpeech                1       1100         16      True   True
   20180501_SYN001_SustainedVowel.wav    SustainedVowel                0       1200         18     False   True
  20180501_SYN001_MentalCondition.wav   MentalCondition                1        900         15      True   True
   20180720_SYN002_SustainedVowel.wav    SustainedVowel                0        200         12     False  False
  20180720_SYN002_MentalCondition.wav   MentalCondition                1        800          8      True  False
       20180720_SYN002_HappyEvent.wav        HappyEvent                1        920         16      True   True
  20180418_SYN003_MentalCondition.wav   MentalCondition                1       1000         20      True

## 3. From recordings to transcripts

In the original analysis (not rerun here):

- about **9000** task recordings
- about **58%** usable after QC
- about **3000** free-speech files transcribed with Whisper

This tutorial does not redistribute audio and does not call Whisper. Linguistic cells below read the invented transcripts in `test_data/`, after applying the same QC keys.

## 4. Linguistic markers

Several exploratory markers were examined (first-person, emotion, absolutist language, and others). Licensed dictionaries are **not** copied here.

The clearest preliminary linguistic signal closer to relapse was **negation ratio**: explicit negation forms (`not`, `n't`, `never`, `cannot`, `couldn't`, `wouldn't`, `shouldn't`, `don't`, …) divided by the number of tokens. In “I am not hopeless”, `not` **is** a negation occurrence.

**Negative emotion** is a secondary lexicon example only. If a negator immediately precedes an emotion word, that emotion hit is skipped (“not hopeless” does not count `hopeless` as negative emotion). That skip does not change the negation ratio.

In [8]:
import re

# Explicit negation forms (presentation marker). Contractions are split in tokenize().
NEGATION_FORMS = {
    "not", "n't", "never", "no", "none", "nobody", "nothing",
    "neither", "nor", "cannot", "can't",
    "won't", "wouldn't", "couldn't", "shouldn't",
    "don't", "doesn't", "didn't", "isn't", "aren't",
    "wasn't", "weren't", "haven't", "hasn't", "hadn't",
}
EMOTION_NEGATORS = {"not", "n't", "never", "no", "cannot", "can't"}
LEXICONS = {
    "first_person": {"i", "me", "my", "mine", "myself"},
    "negative_emotion": {"sad", "tired", "hopeless", "stuck", "wrong", "lost"},
    "positive_emotion": {"happy", "enjoy", "enjoyed", "okay", "stable"},
    "absolutist": {"always", "never", "completely", "nothing", "everything"},
}


def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"n't", " n't ", text)
    return re.findall(r"\b[\w']+\b", text)


def negation_ratio(text):
    """Count explicit negation tokens / n_tokens. Independent of emotion lexicons."""
    words = tokenize(text)
    n = max(len(words), 1)
    count = sum(1 for w in words if w in NEGATION_FORMS)
    return {"n_words": len(words), "negation_count": count, "negation_ratio": count / n}


def lexicon_counts(text, window=1):
    """Lexicon hits. Emotion words immediately after a negator are skipped."""
    words = tokenize(text)
    counts = {name: 0 for name in LEXICONS}
    for i, w in enumerate(words):
        negated = any(words[j] in EMOTION_NEGATORS for j in range(max(0, i - window), i))
        for name, vocab in LEXICONS.items():
            if w in vocab:
                if name == "negative_emotion" and negated:
                    continue
                counts[name] += 1
    n = max(len(words), 1)
    out = {}
    for name, c in counts.items():
        out[f"{name}_count"] = c
        out[f"{name}_ratio"] = c / n
    out["emotion_valence"] = out["positive_emotion_ratio"] - out["negative_emotion_ratio"]
    return out


def extract_features(text):
    feats = negation_ratio(text)
    feats.update(lexicon_counts(text))
    return feats


valid_keys = set(zip(qc.loc[qc["valid"], "patient_id"], qc.loc[qc["valid"], "task"]))
keep = transcripts[transcripts.apply(lambda r: (r["patient_id"], r["task"]) in valid_keys, axis=1)].copy()
print("transcripts before QC filter:", len(transcripts), "; after:", len(keep))
print("dropped:", transcripts.loc[~transcripts.index.isin(keep.index), ["patient_id", "task"]].to_string(index=False))

feat_rows = []
for rec in keep.itertuples(index=False):
    feats = extract_features(rec.transcript)
    feats.update(patient_id=rec.patient_id, date=rec.date, task=rec.task, duration_seconds=rec.duration_seconds)
    feat_rows.append(feats)

features = pd.DataFrame(feat_rows)
show = [
    "patient_id", "task", "n_words",
    "negation_count", "negation_ratio",
    "negative_emotion_count", "positive_emotion_count", "absolutist_count",
]
features[show]

transcripts before QC filter: 16 ; after: 14
dropped: patient_id            task
   SYN_002 MentalCondition
   SYN_004 MentalCondition


In [9]:
raw = "I couldn't sleep and I wouldn't go out. I am not hopeless. I am sad."
print(raw)
print("negation ratio (presentation marker):", negation_ratio(raw))
print("lexicons (hopeless skipped after not; sad counted):", lexicon_counts(raw))
print("expected negation_count >= 3 (couldn't, wouldn't, not)")

I couldn't sleep and I wouldn't go out. I am not hopeless. I am sad.
negation ratio (presentation marker): {'n_words': 17, 'negation_count': 3, 'negation_ratio': 0.17647058823529413}
lexicons (hopeless skipped after not; sad counted): {'first_person_count': 4, 'first_person_ratio': 0.23529411764705882, 'negative_emotion_count': 1, 'negative_emotion_ratio': 0.058823529411764705, 'positive_emotion_count': 0, 'positive_emotion_ratio': 0.0, 'absolutist_count': 0, 'absolutist_ratio': 0.0, 'emotion_valence': -0.058823529411764705}
expected negation_count >= 3 (couldn't, wouldn't, not)


## 5. Scope

This speech section is **exploratory marker analysis**. It is **not** a validated relapse classifier. The synthetic example only shows how negation ratio is counted; it does not reproduce the cohort time-to-relapse result.

The actigraphy contrastive model is in `01_actigraphy_relapse_model.ipynb`. The two streams were analysed separately.